# Latent Space Analysis

## Contributions:
1. **t-SNE clusters** — project all posterior latent states from real episodes into 2D, colored by: velocity magnitude, distance-to-goal, orientation (yaw), and reward value. Tests whether the latent space is *interpretable*.
2. **Cross-interpretation** — correlation heatmap + scatter plots proving that reward correlates with velocity & goal distance as designed. Tests whether the *reward function is genuinely learned*.
3. **Regime distributions** — P(reward | fast/slow) and P(reward | near/far goal) with statistical tests.

### Procedure:
1. Load a trained checkpoint
2. Encode N episodes from the dataset through encoder + RSSM (posterior)
3. Collect latent states + metadata (velocity, goal distance, orientation, reward)
4. Run PCA → t-SNE to project to 2D
5. Generate interpretability and cross-interpretation figures

In [ ]:
import sys
import pickle
from pathlib import Path

import numpy as np
import jax
import jax.numpy as jnp
import h5py
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
import scipy.stats as stats
import ruamel.yaml as yaml
import re

# sklearn for dimensionality reduction
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ── Custom red→yellow→green colormap (vivid, no white midpoint) ───────────────
RYG   = LinearSegmentedColormap.from_list('RYG',   ['#cc0000', '#ffdd00', '#007700'])
RYG_r = LinearSegmentedColormap.from_list('RYG_r', ['#007700', '#ffdd00', '#cc0000'])

# DreamerV3 paths
notebook_dir = Path('/home/maurits-heemskerk/Documents/Uni/Master_Thesis/dreamer_SPOT_implementation/notebooks')
dreamer_dir  = Path('/home/maurits-heemskerk/Documents/Uni/Master_Thesis/dreamer_SPOT_implementation/informed-dreamer')
sys.path.insert(0, str(dreamer_dir))

import dreamerv3
import dreamerv3.embodied as embodied
from dreamerv3 import ninjax as nj

jax.config.update('jax_transfer_guard', 'allow')

print(f'✓ JAX version: {jax.__version__}')
print(f'✓ Devices: {jax.devices()}')
print(f'✓ Imports successful')


## Configuration
Set the run index, number of episodes to encode, and t-SNE parameters here.

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATA_DIR = Path('/media/maurits-heemskerk/69987a47-b840-4db7-9f8b-7cc05f14d09e12/processed_data_NoObs_with_rewards_v1')

# ── Results directory ─────────────────────────────────────────────────────────
results_dir = Path('/home/maurits-heemskerk/Documents/Uni/Master_Thesis/dreamer_results_local_noobs')
available_runs = sorted([d for d in results_dir.iterdir() if d.is_dir()])

print('Available runs:')
for i, d in enumerate(available_runs):
    ok = (d / 'checkpoint.ckpt').exists() and (d / 'config.yaml').exists()
    print(f'  [{i:2d}] {"✓" if ok else "⚠"} {d.name}')

# ── Select run ────────────────────────────────────────────────────────────────
RUN_INDEX = 77  # ← change to select a different run

# ── Encoding settings ─────────────────────────────────────────────────────────
N_EPISODES   = 120    # number of episodes to encode (more = better coverage, slower)
MAX_STEPS_EP = 120   # cap per episode (avoids excessive memory, episodes vary in length)
RANDOM_SEED  = 42    # for t-SNE and sampling reproducibility

# ── t-SNE settings ────────────────────────────────────────────────────────────
TSNE_PERPLEXITY = 30   # typical range: 5-50
PCA_DIM         = 50   # pre-reduce latent dim before t-SNE

# ── Output ────────────────────────────────────────────────────────────────────
SAVE_FIGURES = True

run_path        = available_runs[RUN_INDEX]
CHECKPOINT_PATH = run_path / 'checkpoint.ckpt'
config_file     = run_path / 'config.yaml'
save_dir        = run_path / 'latent_space_results'

assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'
assert config_file.exists(),     f'Config not found: {config_file}'
assert DATA_DIR.exists(),        f'Data directory not found: {DATA_DIR}'

with open(config_file) as f:
    raw_config = yaml.YAML(typ='safe').load(f)

# ── Detect encoder observation keys from config ───────────────────────────────
_enc_mlp = raw_config['encoder']['mlp_keys']
_enc_cnn = raw_config['encoder']['cnn_keys']
_ALL_MLP = ['velocity', 'orientation', 'goal', 'position']
_ALL_CNN = ['image', 'terrain', 'info_terrain']
mlp_keys = [k for k in _ALL_MLP if _enc_mlp and re.search(_enc_mlp, k)]
cnn_keys = [k for k in _ALL_CNN if _enc_cnn and re.search(_enc_cnn, k)]
obs_keys  = mlp_keys + cnn_keys

print(f'\n✓ Run:        {run_path.name}')
print(f'✓ Checkpoint: {CHECKPOINT_PATH.name}')
print(f'✓ Encoder observation keys: {obs_keys}')
print(f'✓ RSSM deter={raw_config["rssm"]["deter"]}  stoch={raw_config["rssm"]["stoch"]}  classes={raw_config["rssm"]["classes"]}')
print(f'\n✓ N_EPISODES={N_EPISODES}  MAX_STEPS_EP={MAX_STEPS_EP}')

## Load Checkpoint & Build World Model

In [ ]:
from dreamerv3.agent import WorldModel

# ── Build embodied config (matching reconstruction notebook exactly) ───────────
_defaults  = embodied.Config(dreamerv3.Agent.configs['defaults'])
_known_keys = set(_defaults.keys()) - {'env'}

def _filter_to_known(cfg, ref):
    out = {}
    for k, v in cfg.items():
        if k not in ref:
            continue
        if isinstance(v, dict) and isinstance(ref[k], dict):
            out[k] = _filter_to_known(v, ref[k])
        else:
            out[k] = v
    return out

_model_config = _filter_to_known(
    {k: v for k, v in raw_config.items() if k in _known_keys},
    dict(_defaults)
)

config = embodied.Config(dreamerv3.Agent.configs['defaults'])
config = config.update(_model_config)
config = config.update({'jax.platform': 'cpu', 'jax.prealloc': False})

# ── Build obs/act space ───────────────────────────────────────────────────────
_SPACE_MAP = {
    'velocity':    embodied.Space(np.float32, (3,)),
    'orientation': embodied.Space(np.float32, (2,)),
    'goal':        embodied.Space(np.float32, (2,)),
    'position':    embodied.Space(np.float32, (2,)),
    'image':       embodied.Space(np.uint8,   (49, 128, 3)),
    'terrain':     embodied.Space(np.float32, (64, 64)),
    'info_terrain':embodied.Space(np.float32, (64, 64)),
}
obs_space = {k: _SPACE_MAP[k] for k in obs_keys if k in _SPACE_MAP}
obs_space['reward']      = embodied.Space(np.float32)
obs_space['is_first']    = embodied.Space(bool)
obs_space['is_last']     = embodied.Space(bool)
obs_space['is_terminal'] = embodied.Space(bool)

act_space = {'action': embodied.Space(np.float32, (3,), -1.0, 1.0), 'reset': embodied.Space(bool)}

# ── Load and remap checkpoint weights ────────────────────────────────────────
with open(CHECKPOINT_PATH, 'rb') as f:
    checkpoint_data = pickle.load(f)

raw_agent_state = checkpoint_data['agent']
agent_state = {}
for k, v in raw_agent_state.items():
    if k.startswith('agent/wm/'):
        agent_state[k.replace('agent/wm/', 'wm/', 1)] = v

print(f'✓ {len(agent_state)} WM weight tensors mapped (from {len(raw_agent_state)} total)')

# ── Create WorldModel ─────────────────────────────────────────────────────────
wm = WorldModel(obs_space, act_space, config, name='wm')
rng_key = jax.random.PRNGKey(RANDOM_SEED)

print(f'✓ WorldModel ready  (deter={config.rssm["deter"]}  stoch={config.rssm["stoch"]}  classes={config.rssm["classes"]})')
print(f'✓ obs_space: {[(k, v.shape) for k, v in obs_space.items() if k not in ("reward","is_first","is_last","is_terminal")]}')

## Batch Encode Episodes

Encode N episodes through the posterior (encoder + RSSM) and collect latent states with metadata.

In [ ]:
h5_files = sorted(DATA_DIR.glob('**/*.h5'))
print(f'Found {len(h5_files)} episode files in dataset')

# Randomly select N_EPISODES (for better coverage, not just first N)
selected_indices = np.random.RandomState(RANDOM_SEED).choice(
    len(h5_files), 
    size=min(N_EPISODES, len(h5_files)), 
    replace=False
)
selected_h5_files = [h5_files[i] for i in selected_indices]
print(f'Will encode {len(selected_h5_files)} random episodes (up to {MAX_STEPS_EP} steps each)\n')

# Storage lists
all_deters      = []   # (T, deter_size)
all_stochs      = []   # (T, stoch*classes)
all_vel_mag     = []   # (T,) — linear velocity magnitude sqrt(vx²+vy²)
all_yaw_rate    = []   # (T,) — angular velocity wz
all_dist_goal   = []   # (T,) — euclidean distance to goal
all_yaw         = []   # (T,) — heading angle in radians (kept for reference)
all_heading_err = []   # (T,) — heading error towards goal (rad), 0=facing goal ±π=away
all_rewards     = []   # (T,)
all_episode     = []   # (T,) — episode index
all_step        = []   # (T,) — timestep within episode

n_encoded = 0
n_failed  = 0

# Helper: extract features from H5 file
def extract_episode(h5_path, max_steps):
    with h5py.File(h5_path, 'r') as f:
        vel_raw = f['observations/velocities'][:].astype(np.float32)  # (T, 6)
        states  = f['observations/state'][:].astype(np.float32)        # (T, 7)
        acts    = f['actions'][:].astype(np.float32)
        rews    = f['rewards'][:].astype(np.float32)

    T = min(len(vel_raw), max_steps)
    vel_raw = vel_raw[:T]
    states  = states[:T]
    acts    = acts[:T]
    rews    = rews[:T]

    # velocity (3,): [vx, vy, wz] — body frame
    velocities = vel_raw[:, [0, 1, 5]].astype(np.float32)

    # Yaw from quaternion
    qx, qy, qz, qw = states[:, 3], states[:, 4], states[:, 5], states[:, 6]
    yaw = np.arctan2(2.0 * (qw * qz + qx * qy), 1.0 - 2.0 * (qy**2 + qz**2))

    # orientation (2,): [cos(yaw), sin(yaw)]
    orientations = np.stack([np.cos(yaw), np.sin(yaw)], axis=-1).astype(np.float32)

    # goal (2,): world-relative displacement to end of episode
    xy_raw        = states[:, :2]
    goal_position = xy_raw[-1]
    goal_relative = (goal_position[None, :] - xy_raw).astype(np.float32)

    # position (2,): zero-centred
    positions = (xy_raw - xy_raw[0]).astype(np.float32)

    # obs_batch
    _data_map = {
        'velocity':    velocities,
        'orientation': orientations,
        'goal':        goal_relative,
        'position':    positions,
    }
    obs_b = {k: np.expand_dims(_data_map[k], 0) for k in obs_keys if k in _data_map}
    obs_b['is_first']    = np.zeros((1, T), dtype=bool)
    obs_b['is_last']     = np.zeros((1, T), dtype=bool)
    obs_b['is_terminal'] = np.zeros((1, T), dtype=bool)
    obs_b['reward']      = np.zeros((1, T), dtype=np.float32)
    obs_b['is_first'][0, 0]  = True
    obs_b['is_last'][0, -1]  = True

    acts_b = np.expand_dims(acts[:, :3], 0)
    acts_prev_b = np.concatenate([np.zeros_like(acts_b[:, :1]), acts_b[:, :-1]], axis=1)

    # Metadata
    vel_mag  = np.sqrt(velocities[:, 0]**2 + velocities[:, 1]**2)  # linear speed
    yaw_rate = velocities[:, 2]                                      # wz
    dist_g   = np.sqrt(goal_relative[:, 0]**2 + goal_relative[:, 1]**2)

    # Heading error: angle between robot's heading and direction to goal
    # 0 = pointing directly at goal, ±π = pointing away
    angle_to_goal = np.arctan2(goal_relative[:, 1], goal_relative[:, 0])
    heading_err   = (angle_to_goal - yaw + np.pi) % (2 * np.pi) - np.pi

    return obs_b, acts_prev_b, vel_mag, yaw_rate, dist_g, yaw, rews, heading_err


# Encode each episode
for ep_idx, h5_path in enumerate(selected_h5_files):
    try:
        obs_b, acts_prev_b, vel_mag, yaw_rate, dist_g, yaw_ep, rews_ep, heading_err_ep = extract_episode(h5_path, MAX_STEPS_EP)
        T = obs_b['is_first'].shape[1]

        # Posterior inference — define closure over this episode's data
        def _encode():
            embed = wm.encoder(obs_b)
            post, _ = wm.rssm.observe(embed, acts_prev_b, obs_b['is_first'])
            return post

        post, _ = nj.pure(_encode)(agent_state, rng_key)

        # Extract latent — squeeze batch dim → (T, dim)
        deter = np.array(post['deter']).squeeze(0)  # (T, deter_size)
        stoch = np.array(post['stoch']).squeeze(0)  # (T, stoch, classes)
        stoch_flat = stoch.reshape(T, -1)             # (T, stoch*classes)

        all_deters.append(deter)
        all_stochs.append(stoch_flat)
        all_vel_mag.append(vel_mag)
        all_yaw_rate.append(yaw_rate)
        all_dist_goal.append(dist_g)
        all_yaw.append(yaw_ep)
        all_heading_err.append(heading_err_ep)
        all_rewards.append(rews_ep)
        all_episode.append(np.full(T, ep_idx, dtype=int))
        all_step.append(np.arange(T))

        n_encoded += 1
        if n_encoded % 10 == 0:
            print(f'  [{n_encoded}/{len(selected_h5_files)}] encoded {T} steps from {h5_path.name}')

    except Exception as e:
        n_failed += 1
        print(f'  ⚠ Episode {ep_idx} failed: {e}')

# Concatenate all into flat arrays
latent_deter     = np.concatenate(all_deters,       axis=0)  # (N_total, deter_size)
latent_stoch     = np.concatenate(all_stochs,       axis=0)  # (N_total, stoch*classes)
vel_mag_all      = np.concatenate(all_vel_mag,      axis=0)  # (N_total,)
yaw_rate_all     = np.concatenate(all_yaw_rate,     axis=0)  # (N_total,)
dist_goal_all    = np.concatenate(all_dist_goal,    axis=0)  # (N_total,)
yaw_all          = np.concatenate(all_yaw,          axis=0)  # (N_total,)
heading_err_all  = np.concatenate(all_heading_err,  axis=0)  # (N_total,)
reward_all       = np.concatenate(all_rewards,      axis=0)  # (N_total,)
episode_all      = np.concatenate(all_episode,      axis=0)  # (N_total,)
step_all         = np.concatenate(all_step,         axis=0)  # (N_total,)

# Combined latent = deter + stoch_flat
latent_full   = np.concatenate([latent_deter, latent_stoch], axis=1)

N_total = len(latent_deter)
print(f'\n✓ Encoded {n_encoded} episodes ({n_failed} failed)')
print(f'✓ Total timesteps: {N_total}')
print(f'✓ latent_deter shape:  {latent_deter.shape}')
print(f'✓ latent_stoch shape:  {latent_stoch.shape}')
print(f'✓ latent_full shape:   {latent_full.shape}')
print(f'\nMetadata ranges:')
print(f'  velocity magnitude:  [{vel_mag_all.min():.3f}, {vel_mag_all.max():.3f}] m/s')
print(f'  distance to goal:    [{dist_goal_all.min():.3f}, {dist_goal_all.max():.3f}] m')
print(f'  heading error:       [{np.degrees(heading_err_all.min()):.1f}, {np.degrees(heading_err_all.max()):.1f}] deg')
print(f'  reward:              [{reward_all.min():.4f}, {reward_all.max():.4f}]')


## PCA → t-SNE Projection

First reduce to `PCA_DIM` dimensions, then apply t-SNE for 2D visualization. Using the full latent `(deter + stoch_flat)` for maximum representativeness.

In [ ]:
print(f'Step 1: PCA {latent_full.shape[1]}D → {PCA_DIM}D ...')
pca = PCA(n_components=PCA_DIM, random_state=RANDOM_SEED)
latent_pca = pca.fit_transform(latent_full)
explained  = pca.explained_variance_ratio_.sum()
print(f'  ✓ PCA done — {explained*100:.1f}% variance explained in {PCA_DIM} components')

print(f'\nStep 2: t-SNE {PCA_DIM}D → 2D  (perplexity={TSNE_PERPLEXITY}, N={N_total}) ...')
print('  (this may take 1-3 minutes for large N)')
tsne = TSNE(
    n_components=2,
    perplexity=TSNE_PERPLEXITY,
    max_iter=1000,
    random_state=RANDOM_SEED,
    verbose=1,
    n_jobs=-1,
)
latent_2d = tsne.fit_transform(latent_pca)

print(f'\n✓ t-SNE complete — latent_2d shape: {latent_2d.shape}')
print(f'  X range: [{latent_2d[:,0].min():.2f}, {latent_2d[:,0].max():.2f}]')
print(f'  Y range: [{latent_2d[:,1].min():.2f}, {latent_2d[:,1].max():.2f}]')

## Figure 1: t-SNE Clusters — Latent Space Interpretability

Four panels: each colors the same 2D projection by a different physical quantity.
Clear gradients / structure prove the latent space encodes interpretable physics.

In [ ]:

# ── Derived quantities ────────────────────────────────────────────────────────
# Clip very large distances for better colormap contrast
dist_clipped = np.clip(dist_goal_all, 0, np.percentile(dist_goal_all, 95))
# Clip velocity magnitude similarly
vel_clipped  = np.clip(vel_mag_all, 0, np.percentile(vel_mag_all, 95))

fig, axes = plt.subplots(2, 3, figsize=(22, 13))

ALPHA  = 0.55
S      = 8

# ── Panel 1: Velocity Magnitude ───────────────────────────────────────────────
ax = axes[0, 0]
sc = ax.scatter(latent_2d[:, 0], latent_2d[:, 1],
                c=vel_clipped, cmap='coolwarm', s=S, alpha=ALPHA, linewidths=0)
cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Linear velocity |v| (m/s)', fontsize=9)
ax.set_title('(a) Colored by Velocity Magnitude', fontweight='bold')
ax.set_xlabel('t-SNE dim 1');  ax.set_ylabel('t-SNE dim 2')
ax.set_xticks([]);  ax.set_yticks([])

# ── Panel 2: Distance to Goal ─────────────────────────────────────────────────
ax = axes[0, 1]
sc = ax.scatter(latent_2d[:, 0], latent_2d[:, 1],
                c=dist_clipped, cmap=RYG_r, s=S, alpha=ALPHA, linewidths=0)
cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Distance to goal (m)', fontsize=9)
ax.set_title('(b) Colored by Distance to Goal', fontweight='bold')
ax.set_xlabel('t-SNE dim 1');  ax.set_ylabel('t-SNE dim 2')
ax.set_xticks([]);  ax.set_yticks([])

# ── Panel 3: Heading Error towards Goal ──────────────────────────────────────
# 0 = robot pointing directly at goal, ±π = pointing away → should correlate with reward
ax = axes[0, 2]
sc = ax.scatter(latent_2d[:, 0], latent_2d[:, 1],
                c=heading_err_all, cmap=RYG_r, s=S, alpha=ALPHA, linewidths=0,
                vmin=-np.pi, vmax=np.pi)
cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Heading error (rad)', fontsize=9)
cbar.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
cbar.set_ticklabels(['-π\n(away)', '-π/2', '0\n(goal)', 'π/2', 'π\n(away)'])
ax.set_title('(c) Colored by Heading Error Towards Goal', fontweight='bold')
ax.set_xlabel('t-SNE dim 1');  ax.set_ylabel('t-SNE dim 2')
ax.set_xticks([]);  ax.set_yticks([])

# ── Panel 4: Reward ───────────────────────────────────────────────────────────
# Clip extreme reward outliers for better color contrast
rew_p5  = np.percentile(reward_all, 5)
rew_p95 = np.percentile(reward_all, 95)
rew_clipped = np.clip(reward_all, rew_p5, rew_p95)

ax = axes[1, 0]
sc = ax.scatter(latent_2d[:, 0], latent_2d[:, 1],
                c=rew_clipped, cmap=RYG, s=S, alpha=ALPHA, linewidths=0)
cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Reward value', fontsize=9)
ax.set_title('(d) Colored by Reward Value', fontweight='bold')
ax.set_xlabel('t-SNE dim 1');  ax.set_ylabel('t-SNE dim 2')
ax.set_xticks([]);  ax.set_yticks([])

# ── Panel 5: Episode Timestep ─────────────────────────────────────────────────
# Does the latent space encode episode progression?
# Gradient from early (0) to late steps → reveals temporal structure if clustered.
ax = axes[1, 1]
sc = ax.scatter(latent_2d[:, 0], latent_2d[:, 1],
                c=step_all, cmap='plasma', s=S, alpha=ALPHA, linewidths=0)
cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Timestep within episode', fontsize=9)
ax.set_title('(e) Colored by Episode Timestep', fontweight='bold')
ax.set_xlabel('t-SNE dim 1');  ax.set_ylabel('t-SNE dim 2')
ax.set_xticks([]);  ax.set_yticks([])

# ── Hide unused 6th panel ─────────────────────────────────────────────────────
axes[1, 2].set_visible(False)

plt.tight_layout()

if SAVE_FIGURES:
    save_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_dir / 'tsne_interpretability.png', dpi=150, bbox_inches='tight')
    print(f'✓ Saved → {save_dir}/tsne_interpretability.png')

plt.show()


## Figure 2: Cross-Interpretation — Reward Function Learning

Do the learned latent states reflect the reward function structure?
- Expect **negative** correlation: velocity ↑ → reward ↓ (safety penalty)
- Expect **negative** correlation: distance-to-goal ↑ → reward ↓ (progress/time penalty)
- If both hold with significance → the model genuinely learned the reward structure

In [ ]:
import seaborn as sns

# Build analysis dataframe
df = pd.DataFrame({
    'velocity':      vel_mag_all,
    'yaw_rate':      np.abs(yaw_rate_all),    # absolute turn rate
    'distance_goal': dist_goal_all,
    'yaw':           yaw_all,
    'reward':        reward_all,
    'episode':       episode_all,
    'step':          step_all,
})

# Compute Pearson correlations with p-values
features = ['velocity', 'yaw_rate', 'distance_goal']
corr_results = {}
for feat in features:
    rho, pval = stats.pearsonr(df[feat], df['reward'])
    corr_results[feat] = {'rho': rho, 'pval': pval}

print('Pearson correlation with reward:')
print(f'{"Feature":20s}  {"rho":>8s}  {"p-value":>12s}  Interpretation')
print('-' * 70)
for feat, res in corr_results.items():
    rho, p = res['rho'], res['pval']
    strength = 'STRONG' if abs(rho) > 0.5 else 'MODERATE' if abs(rho) > 0.3 else 'WEAK'
    interp = '✓ learned' if abs(rho) > 0.3 and p < 0.05 else '✗ not learned'
    print(f'{feat:20s}  {rho:8.3f}  {p:12.2e}  {strength} — {interp}')

# ── Figure 2a: Correlation heatmap ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Cross-Interpretation: Reward Function Learning', fontsize=13, fontweight='bold')

# Heatmap
ax = axes[0]
corr_cols = ['velocity', 'yaw_rate', 'distance_goal', 'reward']
corr_matrix = df[corr_cols].corr()
mask = np.zeros_like(corr_matrix, dtype=bool)
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap=RYG, center=0,
    ax=ax, vmin=-1, vmax=1, square=True, linewidths=0.5,
    cbar_kws={'label': 'Pearson ρ', 'shrink': 0.8},
    xticklabels=['vel', 'yaw\nrate', 'dist\ngoal', 'reward'],
    yticklabels=['velocity', 'yaw rate', 'dist goal', 'reward'],
)
ax.set_title('(a) Correlation Matrix', fontweight='bold')

# Scatter: Velocity vs Reward
ax = axes[1]
hb = ax.hexbin(df['velocity'], df['reward'], gridsize=30, cmap='YlOrRd', mincnt=1)
plt.colorbar(hb, ax=ax, label='Count')
rho = corr_results['velocity']['rho']
p   = corr_results['velocity']['pval']
ax.set_xlabel('Linear Velocity |v| (m/s)', fontsize=10)
ax.set_ylabel('Reward Value', fontsize=10)
ax.set_title('(b) Velocity vs Reward', fontweight='bold')
ax.text(0.5, 0.04,
        f'ρ = {rho:.3f}   p = {p:.1e}',
        transform=ax.transAxes, ha='center', fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.85))
ax.grid(alpha=0.3)

# Scatter: Distance to Goal vs Reward
ax = axes[2]
hb = ax.hexbin(df['distance_goal'], df['reward'], gridsize=30, cmap='YlGnBu', mincnt=1)
plt.colorbar(hb, ax=ax, label='Count')
rho = corr_results['distance_goal']['rho']
p   = corr_results['distance_goal']['pval']
ax.set_xlabel('Distance to Goal (m)', fontsize=10)
ax.set_ylabel('Reward Value', fontsize=10)
ax.set_title('(c) Distance to Goal vs Reward', fontweight='bold')
ax.text(0.5, 0.04,
        f'ρ = {rho:.3f}   p = {p:.1e}',
        transform=ax.transAxes, ha='center', fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.85))
ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.95])

if SAVE_FIGURES:
    save_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_dir / 'cross_interpretation_corr.png', dpi=150, bbox_inches='tight')
    print(f'✓ Saved → {save_dir}/cross_interpretation_corr.png')

plt.show()


## Figure 3: Regime Analysis — P(reward | velocity_regime) and P(reward | distance_regime)

If the reward function is learned, slow states should yield significantly higher rewards than fast states, and near-goal states higher than far-goal states.

In [ ]:
# Define regime bins based on percentiles so they always have data
v33, v67 = np.percentile(df['velocity'], [33, 67])
d33, d67 = np.percentile(df['distance_goal'], [33, 67])

df['vel_regime']  = pd.cut(
    df['velocity'],
    bins=[-np.inf, v33, v67, np.inf],
    labels=['slow (0-33%)', 'medium (33-67%)', 'fast (67-100%)']
)
df['dist_regime'] = pd.cut(
    df['distance_goal'],
    bins=[-np.inf, d33, d67, np.inf],
    labels=['near (0-33%)', 'medium (33-67%)', 'far (67-100%)']
)

# Statistical tests (Welch t-test: slow vs fast, near vs far)
slow_rew = df[df['vel_regime']  == 'slow (0-33%)' ]['reward'].values
fast_rew = df[df['vel_regime']  == 'fast (67-100%)']['reward'].values
near_rew = df[df['dist_regime'] == 'near (0-33%)' ]['reward'].values
far_rew  = df[df['dist_regime'] == 'far (67-100%)']['reward'].values

t_vel,  p_vel  = stats.ttest_ind(slow_rew, fast_rew, equal_var=False)
t_dist, p_dist = stats.ttest_ind(near_rew, far_rew,  equal_var=False)

print('\nRegime mean rewards:')
print(f'  Velocity  — slow: {slow_rew.mean():7.4f}  fast: {fast_rew.mean():7.4f}  diff: {slow_rew.mean()-fast_rew.mean():+.4f}')
print(f'  Distance  — near: {near_rew.mean():7.4f}  far:  {far_rew.mean():7.4f}  diff: {near_rew.mean()-far_rew.mean():+.4f}')
print(f'\nt-tests (Welch):')
print(f'  Velocity (slow vs fast):  t={t_vel:.3f}  p={p_vel:.2e}  → {"✓ SIGNIFICANT" if p_vel < 0.05 else "✗ not significant"}')
print(f'  Distance (near vs far):   t={t_dist:.3f}  p={p_dist:.2e}  → {"✓ SIGNIFICANT" if p_dist < 0.05 else "✗ not significant"}')

# ── Figure 3 ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('P(Reward | Regime) — Does reward separation match reward function design?',
             fontsize=13, fontweight='bold')

COLORS_VEL  = ['#2196F3', '#FFA726', '#F44336']  # blue, orange, red
COLORS_DIST = ['#4CAF50', '#FFA726', '#9C27B0']  # green, orange, purple

# Panel 1: P(reward | velocity regime)
ax = axes[0]
for regime, color in zip(['slow (0-33%)', 'medium (33-67%)', 'fast (67-100%)'], COLORS_VEL):
    subset = df[df['vel_regime'] == regime]['reward']
    ax.hist(subset, bins=40, alpha=0.65, label=f'{regime} (μ={subset.mean():.3f})',
            color=color, density=True, edgecolor='none')
ax.set_xlabel('Reward Value', fontsize=10)
ax.set_ylabel('Probability Density', fontsize=10)
ax.set_title('(a) P(Reward | Velocity Regime)', fontweight='bold')
ax.legend(fontsize=8)
ax.text(0.5, 0.96,
        f'slow vs fast: p = {p_vel:.1e}',
        transform=ax.transAxes, ha='center', va='top', fontsize=9, fontweight='bold',
        color='green' if p_vel < 0.05 else 'red',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.grid(alpha=0.3)

# Panel 2: P(reward | distance regime)
ax = axes[1]
for regime, color in zip(['near (0-33%)', 'medium (33-67%)', 'far (67-100%)'], COLORS_DIST):
    subset = df[df['dist_regime'] == regime]['reward']
    ax.hist(subset, bins=40, alpha=0.65, label=f'{regime} (μ={subset.mean():.3f})',
            color=color, density=True, edgecolor='none')
ax.set_xlabel('Reward Value', fontsize=10)
ax.set_ylabel('Probability Density', fontsize=10)
ax.set_title('(b) P(Reward | Distance Regime)', fontweight='bold')
ax.legend(fontsize=8)
ax.text(0.5, 0.96,
        f'near vs far: p = {p_dist:.1e}',
        transform=ax.transAxes, ha='center', va='top', fontsize=9, fontweight='bold',
        color='green' if p_dist < 0.05 else 'red',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.grid(alpha=0.3)

# Panel 3: Summary box
ax = axes[2]
ax.axis('off')

vel_learned  = abs(corr_results['velocity']['rho']) > 0.3 and corr_results['velocity']['pval'] < 0.05
dist_learned = abs(corr_results['distance_goal']['rho']) > 0.3 and corr_results['distance_goal']['pval'] < 0.05
vel_sep      = p_vel < 0.05
dist_sep     = p_dist < 0.05

summary = (
    f'REWARD FUNCTION LEARNING SUMMARY\n'
    f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n'
    f'Velocity → Reward\n'
    f'  ρ = {corr_results["velocity"]["rho"]:+.3f}   p = {corr_results["velocity"]["pval"]:.1e}\n'
    f'  Regime separation: {"✓ Yes" if vel_sep else "✗ No"}\n'
    f'  → Safety penalty {"learned" if vel_learned else "NOT learned"}\n\n'
    f'Distance → Reward\n'
    f'  ρ = {corr_results["distance_goal"]["rho"]:+.3f}   p = {corr_results["distance_goal"]["pval"]:.1e}\n'
    f'  Regime separation: {"✓ Yes" if dist_sep else "✗ No"}\n'
    f'  → Goal reward {"learned" if dist_learned else "NOT learned"}\n\n'
    f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
    f'Overall: '
    + ('✓✓ STRONG — reward function\n    is genuinely learned'
       if vel_learned and dist_learned
       else '⚠ PARTIAL — some components\n    not learned'
       if vel_learned or dist_learned
       else '✗ WEAK — reward learning\n    not confirmed')
)

ax.text(0.05, 0.95, summary,
        transform=ax.transAxes, fontsize=10, family='monospace',
        va='top', ha='left',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.95, edgecolor='gray'))

plt.tight_layout(rect=[0, 0, 1, 0.95])

if SAVE_FIGURES:
    save_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_dir / 'regime_analysis.png', dpi=150, bbox_inches='tight')
    print(f'✓ Saved → {save_dir}/regime_analysis.png')

plt.show()

## Figure 4: Joint t-SNE — Velocity and Distance Overlaid on Reward

Do the same *latent regions* that have low reward also correspond to high velocity and high goal distance?
This is the key cross-link: if the low-reward cluster coincides with the high-velocity and high-distance cluster, the model has learned a coherent representation of the task.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Joint Analysis: Do low-reward latent regions also correspond\n'
    'to high velocity and high goal distance?',
    fontsize=13, fontweight='bold'
)

x, y = latent_2d[:, 0], latent_2d[:, 1]

# ── Panel 1: Reward ───────────────────────────────────────────────────────────
ax = axes[0]
sc = ax.scatter(x, y, c=rew_clipped, cmap=RYG, s=8, alpha=0.5, linewidths=0)
plt.colorbar(sc, ax=ax, label='Reward', pad=0.01)
ax.set_title('(a) Reward', fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])

# ── Panel 2: Velocity ─────────────────────────────────────────────────────────
ax = axes[1]
sc = ax.scatter(x, y, c=vel_clipped, cmap='coolwarm', s=8, alpha=0.5, linewidths=0)
plt.colorbar(sc, ax=ax, label='Velocity |v| (m/s)', pad=0.01)
ax.set_title('(b) Velocity (expect anti-correlated with reward)', fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])

# ── Panel 3: Heading Error towards Goal ───────────────────────────────────────────────
ax = axes[2]
sc = ax.scatter(x, y, c=heading_err_all, cmap=RYG_r, s=8, alpha=0.5, linewidths=0,
                vmin=-np.pi, vmax=np.pi)
cb = plt.colorbar(sc, ax=ax, label='Heading error (rad)', pad=0.01)
cb.set_ticks([-np.pi, 0, np.pi]); cb.set_ticklabels(['-\u03c0 (away)', '0 (goal)', '\u03c0 (away)'])
ax.set_title('(c) Heading Error (expect correlated with low reward)', fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout(rect=[0, 0, 1, 0.92])

if SAVE_FIGURES:
    save_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_dir / 'tsne_joint_reward_vel_dist.png', dpi=150, bbox_inches='tight')
    print(f'✓ Saved → {save_dir}/tsne_joint_reward_vel_dist.png')

plt.show()

print('\nIf the RED regions in (a) spatially overlap with RED regions in (b) and (c),')
print('the latent space encodes a coherent: low reward ↔ high velocity ↔ far from goal.')
print('This is the strongest evidence of reward function learning in the latent space.')


## Full Statistics Summary

In [ ]:
print('=' * 65)
print(f'LATENT SPACE INTERPRETABILITY — FULL SUMMARY')
print(f'Run: {run_path.name}')
print('=' * 65)

print(f'\nDataset:')
print(f'  Episodes encoded:  {n_encoded}  (failed: {n_failed})')
print(f'  Total timesteps:   {N_total}')
print(f'  Max steps/episode: {MAX_STEPS_EP}')

print(f'\nLatent dimensionality:')
print(f'  deter size:        {latent_deter.shape[1]}')
print(f'  stoch*classes:     {latent_stoch.shape[1]}')
print(f'  full latent:       {latent_full.shape[1]}')
print(f'  PCA → {PCA_DIM}D:         {explained*100:.1f}% variance explained')

print(f'\nCorrelation Analysis (Pearson ρ with reward):')
print(f'  {"Feature":20s}  {"rho":>8s}  {"p-value":>12s}  {"Direction":>10s}  Conclusion')
print(f'  {"-"*70}')
expected_signs = {'velocity': '-', 'yaw_rate': '-', 'distance_goal': '-'}
for feat, res in corr_results.items():
    rho, p = res['rho'], res['pval']
    direction = 'negative' if rho < 0 else 'positive'
    expected  = expected_signs.get(feat, '?')
    matches   = (expected == '-' and rho < 0) or (expected == '+' and rho > 0)
    strength  = 'STRONG' if abs(rho) > 0.5 else 'MODERATE' if abs(rho) > 0.3 else 'WEAK'
    conclusion = f'✓ {strength}' if matches and p < 0.05 else f'✗ unexpected'
    print(f'  {feat:20s}  {rho:8.3f}  {p:12.2e}  {direction:>10s}  {conclusion}')

print(f'\nRegime Separation (Welch t-test):')
print(f'  Velocity slow vs fast:  t={t_vel:6.2f}  p={p_vel:.2e}  μ_slow={slow_rew.mean():.4f}  μ_fast={fast_rew.mean():.4f}')
print(f'  Distance near vs far:   t={t_dist:6.2f}  p={p_dist:.2e}  μ_near={near_rew.mean():.4f}  μ_far={far_rew.mean():.4f}')

print(f'\nFigures saved to: {save_dir}' if SAVE_FIGURES else '')
print('=' * 65)

# Thesis-ready caption text
rho_vel  = corr_results['velocity']['rho']
rho_dist = corr_results['distance_goal']['rho']
print(f'\nSuggested figure caption:')
print(f'"t-SNE projection of posterior latent states (h_t, z_t) from {n_encoded} test')
print(f'episodes ({N_total} timesteps). (a-d) coloring by velocity, distance-to-goal,')
print(f'orientation, and reward reveals {"clear" if abs(rho_vel) > 0.4 or abs(rho_dist) > 0.4 else "limited"}')
print(f'structure in the latent space. Pearson correlation confirms')
print(f'velocity (ρ={rho_vel:.2f}) and goal distance (ρ={rho_dist:.2f}) are negatively')
print(f'correlated with reward (p<0.05), consistent with the designed reward function."')

## Deployment Episode Overlay on t-SNE Projection

Encode real deployment rosbag episodes through the **same** world model and project them into the existing t-SNE space using a k-NN weighted-average mapping (PCA space → 2D). This shows whether the policy visits the latent regions expected by the reward structure.

- Dataset points are shown at low opacity as before.
- Deployment trajectory points are overlaid as bright markers connected by a trajectory line (time → color).
- The same 4-panel layout is reproduced so you can read off velocity / distance / yaw / reward context for each deployment state.

In [ ]:
# ── Select deployment rosbag source ──────────────────────────────────────────
# Point to a run folder that has a 'rosbag/' subdirectory.
# The cell will list all available episode bags inside and let you pick one.

DEPLOY_RUN_DIR = results_dir / 'rewardsv4_2026-05-07_13-26_S16_base'  # ← change me

# Which episode bag to use?  Set to None to use the first available one.
DEPLOY_EP_DIR  = None   # e.g. DEPLOY_RUN_DIR / 'rosbag' / 'episode_20260508_093709'

# k for k-NN projection into existing t-SNE space
TSNE_PROJ_K = 15

# ── Discover bags ─────────────────────────────────────────────────────────────
rosbag_root = DEPLOY_RUN_DIR / 'rosbag'
assert rosbag_root.exists(), f'No rosbag/ directory under {DEPLOY_RUN_DIR}'

ep_dirs = sorted([d for d in rosbag_root.iterdir()
                  if d.is_dir() and any(d.glob('*.db3'))])
assert ep_dirs, f'No .db3 bags found under {rosbag_root}'

print(f'Deployment run: {DEPLOY_RUN_DIR.name}')
print(f'Available episode bags:')
for i, d in enumerate(ep_dirs):
    print(f'  [{i}] {d.name}')

if DEPLOY_EP_DIR is None:
    DEPLOY_EP_DIR = ep_dirs[0]

db3_file = next(DEPLOY_EP_DIR.glob('*.db3'))
print(f'\n✓ Selected: {DEPLOY_EP_DIR.name}')
print(f'  .db3: {db3_file.name}')

In [ ]:
import subprocess, tempfile, textwrap, os

# ── Extraction script (runs under /usr/bin/python3 which has rclpy + rosbag2_py)
_EXTRACT_SCRIPT = textwrap.dedent("""
import sys, json
import numpy as np
import rosbag2_py
from rclpy.serialization import deserialize_message
from nav_msgs.msg import Odometry
from std_msgs.msg import Float32MultiArray

bag_dir  = sys.argv[1]
out_path = sys.argv[2]

reader = rosbag2_py.SequentialReader()
reader.open(
    rosbag2_py.StorageOptions(uri=bag_dir, storage_id='sqlite3'),
    rosbag2_py.ConverterOptions('', ''),
)

odom_list   = []   # [ts, px, py, qx, qy, qz, qw, vx, vy, wz]
action_list = []   # [ts, a0, a1, a2]

while reader.has_next():
    topic, data, ts = reader.read_next()
    if topic == '/odometry':
        msg = deserialize_message(data, Odometry)
        odom_list.append([
            ts,
            msg.pose.pose.position.x, msg.pose.pose.position.y,
            msg.pose.pose.orientation.x, msg.pose.pose.orientation.y,
            msg.pose.pose.orientation.z, msg.pose.pose.orientation.w,
            msg.twist.twist.linear.x, msg.twist.twist.linear.y,
            msg.twist.twist.angular.z,
        ])
    elif topic == '/spot/policy_action_debug':
        msg = deserialize_message(data, Float32MultiArray)
        act = list(msg.data)[:3]
        action_list.append([ts] + act)

if not odom_list:
    print('ERROR: no odometry messages found', file=sys.stderr)
    sys.exit(1)

odom_arr   = np.array(odom_list,   dtype=np.float64)
action_arr = np.array(action_list, dtype=np.float64) if action_list else None

# Sync odom to policy steps (use action timestamps as reference)
if action_arr is not None and len(action_arr) > 1:
    step_idxs = [int(np.argmin(np.abs(odom_arr[:, 0] - ts)))
                 for ts in action_arr[:, 0]]
    step_odom = odom_arr[step_idxs]
    step_acts = action_arr[:, 1:4]
else:
    # No policy_action_debug: downsample odom to ~5 Hz
    dur = (odom_arr[-1, 0] - odom_arr[0, 0]) / 1e9
    step = max(1, int(round(len(odom_arr) / max(dur * 5.0, 1))))
    step_odom = odom_arr[::step]
    step_acts = np.zeros((len(step_odom), 3), dtype=np.float64)

np.savez(
    out_path,
    positions    = step_odom[:, 1:3].astype(np.float32),
    quaternions  = step_odom[:, 3:7].astype(np.float32),  # qx qy qz qw
    velocities   = step_odom[:, 7:10].astype(np.float32), # vx vy wz (body frame)
    actions      = step_acts.astype(np.float32),
)
print(f'steps={len(step_odom)}')
""")

# Write extraction script to a temp file
_script_path = Path(tempfile.gettempdir()) / 'extract_rosbag_deploy.py'
_script_path.write_text(_EXTRACT_SCRIPT)

_npz_path = Path(tempfile.gettempdir()) / 'deploy_episode.npz'

print(f'Extracting {DEPLOY_EP_DIR.name} …')
_result = subprocess.run(
    ['/usr/bin/python3', str(_script_path), str(DEPLOY_EP_DIR), str(_npz_path)],
    capture_output=True, text=True
)
if _result.returncode != 0:
    raise RuntimeError(f'Extraction failed:\n{_result.stderr}')

print(_result.stdout.strip())
if _result.stderr:
    for line in _result.stderr.strip().split('\n'):
        if '[INFO]' not in line:
            print('WARN:', line)

# ── Load extracted data ───────────────────────────────────────────────────────
_npz = np.load(_npz_path)
_pos  = _npz['positions']     # (T, 2) world positions
_quat = _npz['quaternions']   # (T, 4) qx qy qz qw
_vel  = _npz['velocities']    # (T, 3) body-frame vx vy wz
_acts = _npz['actions']       # (T, 3)
T_dep = len(_pos)

# ── Apply same preprocessing as extract_episode ───────────────────────────────
# yaw from quaternion
qx_, qy_, qz_, qw_ = _quat[:,0], _quat[:,1], _quat[:,2], _quat[:,3]
dep_yaw = np.arctan2(2.0*(qw_*qz_ + qx_*qy_), 1.0 - 2.0*(qy_**2 + qz_**2))

dep_orientation = np.stack([np.cos(dep_yaw), np.sin(dep_yaw)], axis=-1).astype(np.float32)
dep_position    = (_pos - _pos[0]).astype(np.float32)
dep_velocity    = _vel.astype(np.float32)                # already body frame
dep_goal_pos    = _pos[-1]                               # last position as goal proxy
dep_goal_rel    = (dep_goal_pos[None,:] - _pos).astype(np.float32)  # (T, 2)

# ── Build obs_b dict (batch=1, T steps) ──────────────────────────────────────
_data_map_dep = {
    'velocity':    dep_velocity,
    'orientation': dep_orientation,
    'goal':        dep_goal_rel,
    'position':    dep_position,
}
dep_obs_b = {k: np.expand_dims(_data_map_dep[k], 0) for k in obs_keys if k in _data_map_dep}
dep_obs_b['is_first']    = np.zeros((1, T_dep), dtype=bool)
dep_obs_b['is_last']     = np.zeros((1, T_dep), dtype=bool)
dep_obs_b['is_terminal'] = np.zeros((1, T_dep), dtype=bool)
dep_obs_b['reward']      = np.zeros((1, T_dep), dtype=np.float32)
dep_obs_b['is_first'][0, 0] = True

dep_acts_b = np.expand_dims(_acts[:, :3], 0)
dep_acts_prev_b = np.concatenate([np.zeros_like(dep_acts_b[:, :1]), dep_acts_b[:, :-1]], axis=1)

# metadata
dep_vel_mag  = np.sqrt(dep_velocity[:,0]**2 + dep_velocity[:,1]**2)
dep_dist_goal = np.sqrt(dep_goal_rel[:,0]**2 + dep_goal_rel[:,1]**2)

print(f'\n✓ Deployment episode preprocessed')
print(f'  Steps: {T_dep}')
print(f'  Velocity magnitude range: [{dep_vel_mag.min():.3f}, {dep_vel_mag.max():.3f}] m/s')
print(f'  Distance to goal range:   [{dep_dist_goal.min():.3f}, {dep_dist_goal.max():.3f}] m')
print(f'  obs_keys used: {obs_keys}')

In [ ]:
# ── Encode deployment episode through world model ─────────────────────────────
def _encode_deploy():
    embed = wm.encoder(dep_obs_b)
    post, _ = wm.rssm.observe(embed, dep_acts_prev_b, dep_obs_b['is_first'])
    return post

dep_post, _ = nj.pure(_encode_deploy)(agent_state, rng_key)

dep_deter = np.array(dep_post['deter']).squeeze(0)  # (T, deter_size)
dep_stoch = np.array(dep_post['stoch']).squeeze(0)  # (T, stoch, classes)
dep_stoch_flat = dep_stoch.reshape(T_dep, -1)
dep_latent_full = np.concatenate([dep_deter, dep_stoch_flat], axis=1)  # (T, full_dim)

print(f'✓ Deployment latent encoded')
print(f'  latent_full shape: {dep_latent_full.shape}')

# ── Project into existing PCA space ───────────────────────────────────────────
dep_latent_pca = pca.transform(dep_latent_full)  # (T, PCA_DIM) — uses fitted PCA
print(f'  latent_pca shape:  {dep_latent_pca.shape}')

# ── Approximate t-SNE projection via k-NN weighted average ───────────────────
# Since sklearn t-SNE has no transform(), approximate by finding the k nearest
# dataset neighbours in PCA space and taking an inverse-distance weighted mean
# of their 2D t-SNE coordinates.
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=TSNE_PROJ_K, algorithm='auto', n_jobs=-1)
knn.fit(latent_pca)   # fit on the full dataset PCA points

knn_dists, knn_idxs = knn.kneighbors(dep_latent_pca)

# Inverse-distance weights (add small epsilon to avoid division by zero)
weights = 1.0 / (knn_dists + 1e-8)
weights /= weights.sum(axis=1, keepdims=True)

# Weighted average of 2D neighbours
dep_latent_2d = np.einsum('nk,nkd->nd', weights, latent_2d[knn_idxs])  # (T, 2)

print(f'  dep_latent_2d shape: {dep_latent_2d.shape}')
print(f'  X range: [{dep_latent_2d[:,0].min():.2f}, {dep_latent_2d[:,0].max():.2f}]')
print(f'  Y range: [{dep_latent_2d[:,1].min():.2f}, {dep_latent_2d[:,1].max():.2f}]')
print(f'\n✓ Deployment trajectory projected into t-SNE space ({TSNE_PROJ_K}-NN interpolation)')

In [ ]:

# ── Overlay plot: t-SNE + deployment trajectory ───────────────────────────────
# Reproduces Figure 1 (4 panels) with the deployment episode overlaid.
# Dataset points: same as before (low opacity, small).
# Deployment points: black markers connected by lines, start = ★  end = ■

from matplotlib.lines import Line2D

# ── Deployment metadata (same normalisations as Figure 1) ─────────────────────
dep_vel_c   = np.clip(dep_vel_mag, 0, np.percentile(vel_mag_all,  95))
dep_dist_c  = np.clip(dep_dist_goal, 0, np.percentile(dist_goal_all, 95))

# Heading error for deployment: angle between robot heading and direction to goal
dep_angle_to_goal  = np.arctan2(dep_goal_rel[:, 1], dep_goal_rel[:, 0])
dep_heading_err    = (dep_angle_to_goal - dep_yaw + np.pi) % (2 * np.pi) - np.pi

# Deployment marker styling
DEP_S       = 25     # marker size
DEP_ALPHA   = 0.85
DEP_COLOR   = 'black'
DEP_LINE_C  = 'black'
DEP_LINE_A  = 0.6
DEP_LINE_W  = 1.2

def _overlay_trajectory(ax, xy2d, label_start=True, label_end=True):
    """Draw trajectory line + black scatter markers."""
    # Trajectory line
    ax.plot(xy2d[:, 0], xy2d[:, 1],
            color=DEP_LINE_C, alpha=DEP_LINE_A, lw=DEP_LINE_W, zorder=3)
    # Black dots
    ax.scatter(xy2d[:, 0], xy2d[:, 1],
               c=DEP_COLOR, s=DEP_S, alpha=DEP_ALPHA,
               edgecolors='white', linewidths=0.4,
               zorder=4, marker='o')
    # Start ★ and End ■
    if label_start:
        ax.scatter(*xy2d[0], marker='*', s=220, c='yellow',
                   edgecolors='black', linewidths=0.8, zorder=5)
    if label_end:
        ax.scatter(*xy2d[-1], marker='s', s=90, c='yellow',
                   edgecolors='black', linewidths=0.8, zorder=5)

x, y = latent_2d[:, 0], latent_2d[:, 1]
dx, dy = dep_latent_2d[:, 0], dep_latent_2d[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle(
    f't-SNE Latent Space  +  Deployment Episode Overlay\n'
    f'Dataset: {n_encoded} eps / {N_total} steps  |  '
    f'Deployment: {DEPLOY_EP_DIR.name}  ({T_dep} steps)',
    fontsize=12, fontweight='bold'
)

# Shared legend elements (start/end markers only)
leg_elems = [
    Line2D([0], [0], marker='*', color='w', markerfacecolor='yellow',
           markersize=10, markeredgecolor='black', label='Deployment start'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='yellow',
           markersize=8,  markeredgecolor='black', label='Deployment end'),
]

# ── Panel 1: Velocity ─────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.scatter(x, y, c=vel_clipped, cmap='coolwarm', s=S, alpha=ALPHA, linewidths=0)
_overlay_trajectory(ax, dep_latent_2d)
plt.colorbar(ax.collections[0], ax=ax, pad=0.01).set_label('Linear velocity |v| (m/s)', fontsize=9)
ax.set_title('(a) Colored by Velocity Magnitude', fontweight='bold')
ax.set_xlabel('t-SNE dim 1');  ax.set_ylabel('t-SNE dim 2')
ax.set_xticks([]);  ax.set_yticks([])

# ── Panel 2: Distance to Goal ─────────────────────────────────────────────────
ax = axes[0, 1]
ax.scatter(x, y, c=dist_clipped, cmap=RYG_r, s=S, alpha=ALPHA, linewidths=0)
_overlay_trajectory(ax, dep_latent_2d)
plt.colorbar(ax.collections[0], ax=ax, pad=0.01).set_label('Distance to goal (m)', fontsize=9)
ax.set_title('(b) Colored by Distance to Goal', fontweight='bold')
ax.set_xlabel('t-SNE dim 1');  ax.set_ylabel('t-SNE dim 2')
ax.set_xticks([]);  ax.set_yticks([])

# ── Panel 3: Heading Error towards Goal ──────────────────────────────────────
ax = axes[1, 0]
ax.scatter(x, y, c=heading_err_all, cmap=RYG_r, s=S, alpha=ALPHA, linewidths=0,
           vmin=-np.pi, vmax=np.pi)
_overlay_trajectory(ax, dep_latent_2d)
cb = plt.colorbar(ax.collections[0], ax=ax, pad=0.01)
cb.set_label('Heading error (rad)', fontsize=9)
cb.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
cb.set_ticklabels(['-π\n(away)', '-π/2', '0\n(goal)', 'π/2', 'π\n(away)'])
ax.set_title('(c) Colored by Heading Error Towards Goal', fontweight='bold')
ax.set_xlabel('t-SNE dim 1');  ax.set_ylabel('t-SNE dim 2')
ax.set_xticks([]);  ax.set_yticks([])

# ── Panel 4: Reward (dataset) + deployment trajectory ─────────────────────────
ax = axes[1, 1]
sc_bg = ax.scatter(x, y, c=rew_clipped, cmap=RYG, s=S, alpha=ALPHA, linewidths=0)
_overlay_trajectory(ax, dep_latent_2d)
plt.colorbar(sc_bg, ax=ax, pad=0.01).set_label('Reward value', fontsize=9)
ax.set_title('(d) Colored by Reward Value', fontweight='bold')
ax.set_xlabel('t-SNE dim 1');  ax.set_ylabel('t-SNE dim 2')
ax.set_xticks([]);  ax.set_yticks([])

# Shared legend (bottom center of fig)
fig.legend(handles=leg_elems, loc='lower center', ncol=2,
           fontsize=8, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout(rect=[0, 0.03, 1, 0.96])

if SAVE_FIGURES:
    save_dir.mkdir(parents=True, exist_ok=True)
    _fname = f'tsne_deployment_overlay_{DEPLOY_EP_DIR.name}.png'
    fig.savefig(save_dir / _fname, dpi=150, bbox_inches='tight')
    print(f'✓ Saved → {save_dir / _fname}')

plt.show()

# ── Quick readout ─────────────────────────────────────────────────────────────
print(f'\nDeployment episode summary:')
print(f'  Steps: {T_dep}')
print(f'  Velocity        mean={dep_vel_mag.mean():.3f}  max={dep_vel_mag.max():.3f} m/s')
print(f'  Dist to goal    mean={dep_dist_goal.mean():.3f}  min={dep_dist_goal.min():.3f} m')
print(f'  Heading error   mean={np.degrees(dep_heading_err.mean()):.1f}°  std={np.degrees(dep_heading_err.std()):.1f}°')
print(f'  Yaw range [{np.degrees(dep_yaw.min()):.0f}°, {np.degrees(dep_yaw.max()):.0f}°]')
